# NPS Long-Horizon Asset Mix Simulator
### JCP 2026 — Track 2 | PFRDA / IAI Joint Conference on Pensions
**Paper:** *Optimal asset mix for NPS under long secular horizons: a liability-aware approach*

---
| Item | Detail |
|------|--------|
| **Method** | Monte Carlo simulation · 10,000 paths |
| **Models** | Regime-switching GBM (equity) · Vasicek (G-Sec) · OU (inflation) |
| **Liability** | IAI LIC 2006-08 mortality · Annuity PV benchmark |
| **AI layer** | Gradient Boosting regime detector · Dynamic de-risking trigger |
| **Output** | Replacement Rate · CVaR-95 · Max Drawdown · HTML report |

> **Run order:** Execute cells top-to-bottom with `Shift+Enter`.  
> Reduce `N_PATHS` to `500` for a quick test run (~15 seconds), then restore to `10_000` for the paper.


> Checked version note: this notebook is designed as a runnable demonstration. The AI detector is trained on synthetic stylised data unless replaced with actual BSE/RBI/CRISIL time series.

## ① Install dependencies & imports

In [ ]:
# ── Install (safe to re-run; skips if already installed) ─────────────────────
import sys
!{sys.executable} -m pip install numpy pandas matplotlib scipy scikit-learn openpyxl ipywidgets --quiet

# ── Standard imports ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter
from matplotlib.gridspec import GridSpec
from scipy.stats import norm
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import base64, io, os, datetime, warnings
warnings.filterwarnings("ignore")

# ── Jupyter display helpers ───────────────────────────────────────────────────
from IPython.display import display, HTML, IFrame
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

np.random.seed(42)
print("✅  All imports successful.")


## ② Configuration — edit parameters here
All capital market, model, and simulation parameters live in the `CFG` dictionary.  
**Blue cells = inputs you can change.**

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  EDIT THESE PARAMETERS TO CUSTOMISE THE SIMULATION           ║
# ╚══════════════════════════════════════════════════════════════╝

N_PATHS = 500           # Quick Jupyter run. Change to 10_000 for final paper run.

CFG = dict(
    # ── Simulation ────────────────────────────────────────────────
    n_paths        = N_PATHS,
    dt             = 1/12,          # monthly time step
    retire_age     = 60,
    max_age        = 90,            # terminal age for annuity tail

    # ── Contribution rates ────────────────────────────────────────
    contrib_govt   = 0.24,          # 10% subscriber + 14% employer
    contrib_corp   = 0.20,          # 10% + 10%

    # ── Equity (regime-switching GBM) ─────────────────────────────
    mu_equity_bull   = 0.135,       # bull regime drift  (BSE Sensex 2000-23)
    mu_equity_bear   = -0.080,      # bear regime drift
    sigma_equity_bull= 0.180,       # bull volatility
    sigma_equity_bear= 0.380,       # bear volatility
    p_bull_to_bear   = 0.060,       # monthly transition probability
    p_bear_to_bull   = 0.250,

    # ── Corporate bonds ───────────────────────────────────────────
    mu_corp     = 0.080,
    sigma_corp  = 0.045,

    # ── Alternative assets / InvIT ────────────────────────────────
    mu_invit    = 0.105,            # ~180bps illiquidity premium over G-Sec
    sigma_invit = 0.120,

    # ── Vasicek interest-rate model (G-Sec) ───────────────────────
    kappa   = 0.18,                 # speed of mean reversion
    theta   = 0.072,                # long-run mean yield (7.2%)
    sigma_r = 0.012,                # rate volatility
    r0      = 0.072,                # initial short rate
    rho_eq_r= -0.18,                # equity–rate correlation

    # ── Ornstein-Uhlenbeck inflation ──────────────────────────────
    kappa_pi= 0.20,
    theta_pi= 0.055,                # RBI medium-term CPI target
    sigma_pi= 0.015,
    pi0     = 0.055,

    # ── Salary & liability ────────────────────────────────────────
    g_salary    = 0.020,            # real salary growth
    target_rr   = 0.50,            # 50% replacement-rate target

    # ── Optimisation ──────────────────────────────────────────────
    lambda_risk  = 0.50,            # mean-CVaR risk-aversion weight
    cvar_alpha   = 0.95,

    # ── AI overlay ────────────────────────────────────────────────
    stress_threshold = 0.65,        # trigger de-risking above this
    calm_threshold   = 0.35,        # restore equity below this
    derisking_step   = 0.10,        # equity reduction on trigger (pp)
)

SCENARIOS = {
    "Base Case"           : dict(mu_eq=0.135, mu_gsec=0.072, theta=0.072),
    "High Inflation"      : dict(mu_eq=0.110, mu_gsec=0.085, theta=0.085),
    "Rising Rates +150bps": dict(mu_eq=0.100, mu_gsec=0.090, theta=0.090),
    "Market Stress"       : dict(mu_eq=-0.08, mu_gsec=0.080, theta=0.080),
    "Prolonged Low Growth": dict(mu_eq=0.070, mu_gsec=0.055, theta=0.055),
    "Bull Case"           : dict(mu_eq=0.170, mu_gsec=0.075, theta=0.075),
}

ENTRY_AGES   = [25, 35, 45]
STRAT_LABELS = {
    "current_75": "Current NPS (75% E)",
    "proposed"  : "Proposed (85% E + InvIT)",
    "ai_dynamic": "AI Dynamic Glide Path",
}

# ── Dark-theme colour palette ─────────────────────────────────────────────────
P = dict(
    equity ="■ #E8A838", gsec="■ #4A90D9", corp="■ #5CB85C",
    invit  ="■ #9B59B6",
    current="#E74C3C", proposed="#2ECC71", ai="#F39C12",
    bg     ="#0D1117",  surface="#161B22", border="#30363D",
    text   ="#E6EDF3",  subtext="#8B949E",
)
COL = dict(
    eq="#E8A838", gs="#4A90D9", cb="#5CB85C", alt="#9B59B6",
    cur="#E74C3C", prop="#2ECC71", ai="#F39C12",
)

plt.rcParams.update({
    "figure.facecolor":"#0D1117","axes.facecolor":"#161B22",
    "axes.edgecolor":"#30363D","axes.labelcolor":"#E6EDF3",
    "xtick.color":"#8B949E","ytick.color":"#8B949E",
    "text.color":"#E6EDF3","grid.color":"#30363D",
    "grid.linestyle":"--","grid.alpha":0.5,
    "legend.facecolor":"#161B22","legend.edgecolor":"#30363D",
    "font.family":"monospace",
})

print(f"✅  Configuration loaded  |  n_paths = {N_PATHS:,}  |  retire_age = {CFG['retire_age']}")


## ③ Mortality table — IAI LIC 2006-08 (projected to 2026)

In [ ]:
def build_mortality():
    """Makeham-Gompertz fit to IAI LIC 2006-08 Ultimate qx values,
    with 1.5% p.a. generational improvement from 2008 base to 2026."""
    ages = np.arange(0, 101)
    A, B, c = 0.00022, 0.0000027, 1.124
    qx = np.clip(A + B * (c ** ages), 0, 1) * ((1 - 0.015) ** 18)
    lx = np.zeros(101); lx[0] = 100_000
    for i in range(1, 101):
        lx[i] = lx[i-1] * (1 - qx[i-1])
    return qx, lx

QX, LX = build_mortality()

def tpx(x, t):
    """Probability of surviving t years from exact age x."""
    return LX[min(x+t, 100)] / LX[x] if LX[x] > 0 else 0.0

def annuity_pv(x, i, max_age=None):
    """Actuarial present value of a level whole-life annuity-due at age x."""
    ma = max_age or CFG["max_age"]
    v  = 1 / (1 + i)
    return sum(v**t * tpx(x, t) for t in range(ma - x + 1))

# ── Preview mortality table ───────────────────────────────────────────────────
sel = [25,30,35,40,45,50,55,60,65,70,75,80]
mort_preview = pd.DataFrame({
    "Age x"         : sel,
    "qx"            : [f"{QX[a]:.6f}" for a in sel],
    "lx (per 100k)" : [f"{LX[a]:,.0f}"  for a in sel],
    "₁₀pₓ"          : [f"{tpx(a,10):.4f}" for a in sel],
    "ä(x) i=7.2%"   : [f"{annuity_pv(a, 0.072):.3f}" for a in sel],
})
display(mort_preview.style
    .set_caption("IAI LIC 2006-08 Mortality | 1.5%/yr improvement to 2026")
    .set_properties(**{"font-size":"11px","text-align":"right"})
    .set_table_styles([
        {"selector":"caption","props":[("font-weight","bold"),("font-size","13px")]},
        {"selector":"th","props":[("background","#1F3864"),("color","#E8D5A0"),
                                   ("text-align","center"),("padding","6px 12px")]},
        {"selector":"tr:nth-child(even)","props":[("background","#EBF3FB")]},
    ]))


## ④ Capital market simulation models

In [ ]:
def _cfg(c=None):
    """Return local config dict; needed so scenario overrides actually flow into simulations."""
    return CFG if c is None else c

def sim_vasicek(n, steps, c=None):
    """Short-rate paths via Vasicek model. Returns (n, steps+1) array."""
    c = _cfg(c)
    r = np.zeros((n, steps+1)); r[:,0] = c["r0"]
    dW = np.random.normal(0, np.sqrt(c["dt"]), (n, steps))
    for t in range(steps):
        r[:,t+1] = (r[:,t]
                    + c["kappa"] * (c["theta"] - r[:,t]) * c["dt"]
                    + c["sigma_r"] * dW[:,t])
    return np.maximum(r, 0.005)

def sim_inflation(n, steps, c=None):
    """OU inflation paths. Returns (n, steps+1) array."""
    c = _cfg(c)
    pi = np.zeros((n, steps+1)); pi[:,0] = c["pi0"]
    dW = np.random.normal(0, np.sqrt(c["dt"]), (n, steps))
    for t in range(steps):
        pi[:,t+1] = (pi[:,t]
                     + c["kappa_pi"] * (c["theta_pi"] - pi[:,t]) * c["dt"]
                     + c["sigma_pi"] * dW[:,t])
    return np.maximum(pi, 0.01)

def sim_equity(n, steps, rate_paths=None, c=None):
    """Regime-switching GBM equity returns. Returns monthly returns and regime path."""
    c = _cfg(c)
    dt = c["dt"]
    dZ_eq = np.random.normal(0, np.sqrt(dt), (n, steps))
    dZ_r  = np.random.normal(0, np.sqrt(dt), (n, steps))
    rho   = c["rho_eq_r"]
    dW_eq = rho * dZ_r + np.sqrt(1 - rho**2) * dZ_eq

    regime = np.zeros((n, steps), dtype=int)
    for t in range(1, steps):
        tr = np.random.random(n)
        regime[:,t] = np.where(regime[:,t-1]==0,
                               np.where(tr < c["p_bull_to_bear"], 1, 0),
                               np.where(tr < c["p_bear_to_bull"], 0, 1))
    mu    = np.where(regime==0, c["mu_equity_bull"],    c["mu_equity_bear"])
    sigma = np.where(regime==0, c["sigma_equity_bull"], c["sigma_equity_bear"])
    return np.exp((mu - 0.5*sigma**2)*dt + sigma*dW_eq) - 1, regime

def sim_lognormal(n, steps, mu_ann, sig_ann, c=None):
    """Simple monthly normal return approximation for corp bonds / InvIT."""
    c = _cfg(c)
    dt  = c["dt"]
    return np.random.normal(mu_ann*dt, sig_ann*np.sqrt(dt), (n, steps))

print("✅  Market model functions defined. Scenario overrides now flow into the simulation.")


## ⑤ Glide path definitions & visualisation

In [ ]:
def glide_weights(strategy, age):
    """Asset weights (E, C, G, Alt) for a given strategy and subscriber age."""
    if strategy == "current_75":
        wE = 0.75 if age <= 35 else (0.15 if age >= 55
             else 0.75 - (age-35)*(0.60/20))
        wC, wAlt = 0.10, 0.05
        wG = max(0, 1 - wE - wC - wAlt)
    elif strategy in ("proposed", "ai_dynamic"):
        if   age <= 34: wE, wAlt = 0.85, 0.10
        elif age <= 44: wE, wAlt = 0.70, 0.10
        elif age <= 49: wE, wAlt = 0.55, 0.15
        elif age <= 54: wE, wAlt = 0.35, 0.20
        else:           wE, wAlt = 0.15, 0.20
        wC = 0.05; wG = max(0, 1 - wE - wC - wAlt)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")
    return dict(E=wE, C=wC, G=wG, Alt=wAlt)

# ── Glide path chart ──────────────────────────────────────────────────────────
ages = np.arange(25, 61)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("NPS Asset Allocation Glide Paths", fontsize=13, y=1.01)

for ax, strat, title, border_col in zip(
        axes,
        ["current_75", "proposed"],
        ["Current NPS — Active Choice / LC75", "Proposed Revised Glide Path"],
        [COL["cur"], COL["prop"]]):
    wE   = [glide_weights(strat, a)["E"]   for a in ages]
    wC   = [glide_weights(strat, a)["C"]   for a in ages]
    wG   = [glide_weights(strat, a)["G"]   for a in ages]
    wAlt = [glide_weights(strat, a)["Alt"] for a in ages]
    ax.stackplot(ages,
                 [np.array(wE), np.array(wC), np.array(wG), np.array(wAlt)],
                 labels=["Equity (E)","Corp Bonds (C)","G-Sec (G)","Alt/InvIT"],
                 colors=[COL["eq"], COL["cb"], COL["gs"], COL["alt"]],
                 alpha=0.85)
    ax.set_title(title, color=border_col, fontsize=11, pad=8)
    ax.set_xlabel("Subscriber Age"); ax.set_ylabel("Allocation")
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v,_: f"{v*100:.0f}%"))
    ax.set_xlim(25, 60); ax.set_ylim(0, 1)
    ax.legend(loc="upper right", fontsize=8); ax.grid(True, alpha=0.3)
    for spine in ax.spines.values(): spine.set_edgecolor(border_col)

plt.tight_layout()
plt.savefig("fig_glide_paths.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("✅  Glide path chart saved → fig_glide_paths.png")


## ⑥ AI regime detector — Gradient Boosting Classifier

In [ ]:
class RegimeDetector:
    """
    Gradient Boosting classifier for bear/bull regime detection.
    Features: 12-month equity return, 12-month volatility, yield-curve slope,
              credit spread, P/E ratio.
    Training: synthetic data calibrated to Indian market stylised facts.
    Production: replace _train_on_synthetic() with real BSE / RBI time series.
    """
    def __init__(self):
        self.model  = GradientBoostingClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05,
            subsample=0.8, random_state=42)
        self.scaler = StandardScaler()
        self._train()

    def _train(self):
        np.random.seed(7)
        n = 2000
        y = np.random.binomial(1, 0.20, n)   # ~20% stress periods
        X = np.column_stack([
            np.where(y, np.random.normal(-0.15,0.10,n), np.random.normal(0.13,0.08,n)),
            np.where(y, np.random.normal( 0.32,0.06,n), np.random.normal(0.16,0.03,n)),
            np.where(y, np.random.normal(-0.50,0.30,n), np.random.normal(0.80,0.35,n)),
            np.where(y, np.random.normal( 2.50,0.50,n), np.random.normal(1.00,0.30,n)),
            np.where(y, np.random.normal(16.0, 3.0, n), np.random.normal(22.0, 4.0,n)),
        ])
        Xs = self.scaler.fit_transform(X)
        self.model.fit(Xs, y)
        oos_prob = self.model.predict_proba(Xs[-400:])[:,1]
        auc = roc_auc_score(y[-400:], oos_prob)
        print(f"  Regime Classifier  |  OOS AUC = {auc:.3f}  "
              f"|  Feature importances: {self.model.feature_importances_.round(3)}")

    def stress_prob_batch(self, features_2d):
        """Vectorised: features_2d shape (n_paths, 5). Returns (n_paths,) stress probs."""
        return self.model.predict_proba(self.scaler.transform(features_2d))[:,1]

print("Training AI Regime Detector …")
DETECTOR = RegimeDetector()
print("✅  Regime detector ready.")


## ⑦ Monte Carlo simulation engine

In [ ]:
def run_simulation(entry_age, strategy, scenario=None):
    """
    Full Monte Carlo simulation for one cohort and strategy.

    Parameters
    ----------
    entry_age : int          Subscriber entry age (25, 35, or 45)
    strategy  : str          'current_75' | 'proposed' | 'ai_dynamic'
    scenario  : dict | None  Optional capital-market overrides

    Returns
    -------
    dict with keys: replacement_rate, funding_ratio, cvar_95,
                    median_rr, p5_rr, p95_rr, drawdown_max,
                    regime_paths, salary_terminal
    """
    c = dict(CFG)
    if scenario:
        # Important: these now affect the simulation because c is passed into the model functions.
        c["mu_equity_bull"] = scenario.get("mu_eq",   c["mu_equity_bull"])
        c["theta"]          = scenario.get("theta",   c["theta"])
        c["r0"]             = scenario.get("mu_gsec", c["r0"])

    n      = c["n_paths"]
    years  = c["retire_age"] - entry_age
    steps  = int(years / c["dt"])
    dt     = c["dt"]

    # ── Simulate market paths ─────────────────────────────────────────────────
    rate_paths          = sim_vasicek(n, steps, c)
    eq_ret, reg_paths   = sim_equity(n, steps, rate_paths, c)
    cb_ret              = sim_lognormal(n, steps, c["mu_corp"],  c["sigma_corp"], c)
    alt_ret             = sim_lognormal(n, steps, c["mu_invit"], c["sigma_invit"], c)
    pi_paths            = sim_inflation(n, steps, c)

    # G-Sec total return: coupon income + duration-adjusted price change
    dur_gsec = 8.0
    gsec_ret = rate_paths[:,1:]*dt - dur_gsec * np.diff(rate_paths, axis=1)

    # ── Salary paths ──────────────────────────────────────────────────────────
    salary = np.zeros((n, steps+1)); salary[:,0] = 1.0
    for t in range(steps):
        salary[:,t+1] = salary[:,t] * (1 + (c["g_salary"] + pi_paths[:,t])*dt)

    # ── Portfolio accumulation ────────────────────────────────────────────────
    corpus = np.zeros((n, steps+1))
    eq_rolling  = np.zeros(n)
    vol_rolling = np.full(n, 0.18)
    ai_adj      = np.zeros(n)   # AI equity adjustment (negative = reduction)

    for t in range(steps):
        age_now = entry_age + t * dt
        bw = glide_weights(strategy, int(age_now))
        wE, wC, wG, wAlt = bw["E"], bw["C"], bw["G"], bw["Alt"]

        # ── AI overlay: annual rebalancing ────────────────────────────────────
        if strategy == "ai_dynamic" and t % 12 == 0 and t > 12:
            slope  = rate_paths[:,t] - rate_paths[:,max(0,t-12)]
            spread = np.full(n, 1.0)
            pe     = np.full(n, 22.0) * (1 - eq_rolling)
            feats  = np.column_stack([eq_rolling, vol_rolling, slope, spread, pe])
            sp     = DETECTOR.stress_prob_batch(feats)
            ai_adj = np.where(sp > c["stress_threshold"], -c["derisking_step"],
                     np.where(sp < c["calm_threshold"],    0.0, ai_adj))

        # Move any AI equity reduction into G-Sec. This keeps weights invested and avoids
        # silently creating an unintended cash position when base G-Sec weight is zero.
        wE_eff = np.clip(wE + ai_adj, 0, 1)
        shift_to_gsec = np.maximum(wE - wE_eff, 0)
        wG_eff = np.clip(wG + shift_to_gsec, 0, 1)

        port_ret = (wE_eff * eq_ret[:,t] + wC * cb_ret[:,t]
                    + wG_eff * gsec_ret[:,t] + wAlt * alt_ret[:,t])
        contrib  = salary[:,t] * c["contrib_govt"] * dt
        corpus[:,t+1] = corpus[:,t] * (1 + port_ret) + contrib

        if strategy == "ai_dynamic":
            eq_rolling  = 0.97*eq_rolling  + 0.03*eq_ret[:,t]/dt
            vol_rolling = 0.97*vol_rolling + 0.03*np.abs(eq_ret[:,t])/np.sqrt(dt)

    # ── Terminal replacement rate ─────────────────────────────────────────────
    mean_rate  = float(np.mean(rate_paths[:,-1]))
    apv        = annuity_pv(c["retire_age"], mean_rate, c["max_age"])
    liability  = salary[:,-1] * c["target_rr"] * apv
    rr         = corpus[:,-1] / liability
    fr         = corpus / np.maximum(salary * c["target_rr"] * apv, 1e-9)

    # CVaR: mean of the worst 5% replacement-rate outcomes, shown as a positive RR multiple.
    q          = np.percentile(rr, (1-c["cvar_alpha"])*100)
    tail_rr    = rr[rr <= q]
    cvar       = float(np.mean(tail_rr)) if len(tail_rr) else float(q)

    max_dd     = float(np.mean(np.max((np.maximum.accumulate(fr,axis=1)-fr)
                                       / np.maximum(np.maximum.accumulate(fr,axis=1),1e-9),
                                       axis=1)))
    return dict(
        replacement_rate = rr,
        funding_ratio    = fr,
        regime_paths     = reg_paths,
        salary_terminal  = salary[:,-1],
        cvar_95          = cvar,
        median_rr        = float(np.median(rr)),
        p5_rr            = float(np.percentile(rr, 5)),
        p95_rr           = float(np.percentile(rr, 95)),
        drawdown_max     = max_dd,
    )

print("✅  Simulation engine defined.")


## ⑧ Run all simulations
> ⏱ Expect **3–8 min** at 10,000 paths. Reduce `N_PATHS = 500` in Cell ② for a fast test.

In [ ]:
import time
RESULTS = {}

print("─"*65)
print(f"  Running Monte Carlo  |  {CFG['n_paths']:,} paths  |  3 cohorts × 3 strategies")
print("─"*65)

for age in ENTRY_AGES:
    RESULTS[age] = {}
    for strat, label in STRAT_LABELS.items():
        t0 = time.time()
        RESULTS[age][strat] = run_simulation(age, strat)
        elapsed = time.time() - t0
        r = RESULTS[age][strat]
        print(f"  Age {age:2d} | {label:<28} | "
              f"Median RR = {r['median_rr']:.2f}x  "
              f"CVaR = {r['cvar_95']:.2f}x  "
              f"({elapsed:.0f}s)")

print()
print("─"*65)
print("  Scenario analysis  (Age 25 · Proposed strategy)")
print("─"*65)
RESULTS["scenarios"] = {}
for sc_name, sc_cfg in SCENARIOS.items():
    t0 = time.time()
    RESULTS["scenarios"][sc_name] = run_simulation(25, "proposed", scenario=sc_cfg)
    elapsed = time.time() - t0
    r = RESULTS["scenarios"][sc_name]
    print(f"  {sc_name:<28} | Median RR = {r['median_rr']:.2f}x  ({elapsed:.0f}s)")

print()
print("✅  All simulations complete.")


## ⑨ Results tables

In [ ]:
# ── Table A: Cohort × Strategy summary ───────────────────────────────────────
rows = []
for age in ENTRY_AGES:
    for strat, label in STRAT_LABELS.items():
        r = RESULTS[age][strat]
        rows.append({
            "Entry Age" : age,
            "Strategy"  : label,
            "Median RR" : r["median_rr"],
            "P5 RR"     : r["p5_rr"],
            "P95 RR"    : r["p95_rr"],
            "CVaR 95%"  : r["cvar_95"],
            "Max DD %"  : r["drawdown_max"]*100,
        })
df_cohort = pd.DataFrame(rows)

# ── Table B: Scenario summary ─────────────────────────────────────────────────
sc_rows = []
for sc in SCENARIOS:
    r = RESULTS["scenarios"][sc]
    sc_rows.append({
        "Scenario"  : sc,
        "Median RR" : r["median_rr"],
        "P5 RR"     : r["p5_rr"],
        "P95 RR"    : r["p95_rr"],
        "CVaR 95%"  : r["cvar_95"],
        "Max DD %"  : r["drawdown_max"]*100,
    })
df_scenarios = pd.DataFrame(sc_rows)

# ── Styled display ────────────────────────────────────────────────────────────
def style_rr(v):
    """Colour-code replacement rate cells."""
    if isinstance(v, float):
        if v >= 1.5:  return "color:#2ECC71;font-weight:bold"
        if v >= 1.0:  return "color:#F39C12;font-weight:bold"
        return "color:#E74C3C;font-weight:bold"
    return ""

TH_STYLE = [
    {"selector":"th","props":[("background","#1F3864"),("color","#E8D5A0"),
                               ("text-align","center"),("padding","7px 14px"),
                               ("font-size","11px")]},
    {"selector":"tr:nth-child(even)","props":[("background","#EBF3FB")]},
    {"selector":"caption","props":[("font-size","13px"),("font-weight","bold"),
                                    ("text-align","left"),("padding","4px")]},
]

print("Table A — Cohort × Strategy Simulation Results")
display(df_cohort.style
    .format({"Median RR":"{:.2f}x","P5 RR":"{:.2f}x","P95 RR":"{:.2f}x",
             "CVaR 95%":"{:.2f}x","Max DD %":"{:.1f}%"})
    .applymap(style_rr, subset=["Median RR","P5 RR"])
    .set_caption("Table A — Monte Carlo Results: 10,000 paths | Replacement Rate = corpus ÷ annuity liability")
    .set_table_styles(TH_STYLE)
    .hide(axis="index"))

print()
print("Table B — Scenario Analysis (Age 25 · Proposed Strategy)")
display(df_scenarios.style
    .format({"Median RR":"{:.2f}x","P5 RR":"{:.2f}x","P95 RR":"{:.2f}x",
             "CVaR 95%":"{:.2f}x","Max DD %":"{:.1f}%"})
    .applymap(style_rr, subset=["Median RR","P5 RR"])
    .set_caption("Table B — Scenario Analysis: six macroeconomic regimes")
    .set_table_styles(TH_STYLE)
    .hide(axis="index"))


## ⑩ Monte Carlo fan charts — funding ratio trajectories

In [ ]:
def plot_fan(age):
    years = CFG["retire_age"] - age
    x     = np.linspace(0, years, int(years/CFG["dt"]) + 1)
    strats = ["current_75","proposed","ai_dynamic"]
    titles = ["Current NPS (75% E)","Proposed (85% E + InvIT)","AI Dynamic"]
    colors = [COL["cur"], COL["prop"], COL["ai"]]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
    fig.suptitle(f"Funding Ratio Trajectories — Entry Age {age}  "
                 f"({years}-year horizon · {CFG['n_paths']:,} paths)",
                 fontsize=12, y=1.01)

    for ax, strat, title, col in zip(axes, strats, titles, colors):
        fr  = RESULTS[age][strat]["funding_ratio"]
        p5  = np.percentile(fr, 5,  axis=0)
        p25 = np.percentile(fr, 25, axis=0)
        p50 = np.percentile(fr, 50, axis=0)
        p75 = np.percentile(fr, 75, axis=0)
        p95 = np.percentile(fr, 95, axis=0)

        ax.fill_between(x, p5,  p95, alpha=0.10, color=col)
        ax.fill_between(x, p25, p75, alpha=0.22, color=col)
        ax.plot(x, p50, color=col, lw=2.2, label=f"Median {np.median(fr[:,-1]):.2f}x")
        ax.axhline(1.0, color="white", lw=1.0, ls=":", alpha=0.7, label="Target 1.0x")
        ax.set_title(title, color=col, fontsize=10, pad=6)
        ax.set_xlabel("Years from entry")
        ax.set_ylabel("Funding Ratio")
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v,_: f"{v:.1f}x"))
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
        for sp in ax.spines.values(): sp.set_edgecolor(col)

    plt.tight_layout()
    plt.savefig(f"fig_fan_age{age}.png", dpi=150,
                bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()

for age in ENTRY_AGES:
    plot_fan(age)
print("✅  Fan charts saved.")


## ⑪ Replacement rate distribution — violin plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=False)
fig.suptitle("Replacement Rate Distributions — 10,000 Paths per Strategy",
             fontsize=12, y=1.01)

for ax, age in zip(axes, ENTRY_AGES):
    data   = [np.clip(RESULTS[age][s]["replacement_rate"], 0, 7)
              for s in ["current_75","proposed","ai_dynamic"]]
    colors = [COL["cur"], COL["prop"], COL["ai"]]
    parts  = ax.violinplot(data, positions=[1,2,3], showmedians=True,
                            showextrema=False, widths=0.65)
    for pc, col in zip(parts["bodies"], colors):
        pc.set_facecolor(col); pc.set_alpha(0.52)
    parts["cmedians"].set_color("white"); parts["cmedians"].set_linewidth(2.2)

    ax.axhline(1.0, color="white", ls=":", lw=1.0, alpha=0.7)
    ax.set_xticks([1,2,3])
    ax.set_xticklabels(["Current","Proposed","AI Dyn"], rotation=18, ha="right")
    ax.set_title(f"Entry Age {age}", fontsize=11)
    ax.set_ylabel("Replacement Rate")
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v,_: f"{v:.1f}x"))
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("fig_rr_distributions.png", dpi=150,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print("✅  Distribution chart saved → fig_rr_distributions.png")


## ⑫ Scenario analysis chart

In [ ]:
sc_names = list(SCENARIOS.keys())
medians  = [RESULTS["scenarios"][s]["median_rr"]  for s in sc_names]
p5s      = [RESULTS["scenarios"][s]["p5_rr"]      for s in sc_names]
cvars    = [abs(RESULTS["scenarios"][s]["cvar_95"]) for s in sc_names]

x = np.arange(len(sc_names)); w = 0.26
fig, ax = plt.subplots(figsize=(12, 5))

b1 = ax.bar(x-w,   medians, w, label="Median RR",   color=COL["prop"], alpha=0.85)
b2 = ax.bar(x,     p5s,    w, label="P5 RR",        color=COL["eq"],   alpha=0.85)
b3 = ax.bar(x+w,   cvars,  w, label="CVaR 95%",     color=COL["cur"],  alpha=0.85)

ax.axhline(1.0, color="white", ls=":", lw=1.3, alpha=0.7, label="Target = 1.0x")
ax.set_xticks(x)
ax.set_xticklabels(sc_names, rotation=22, ha="right")
ax.set_ylabel("Replacement Rate / CVaR")
ax.set_title("Scenario Analysis — Entry Age 25 · Proposed Strategy", fontsize=12)
ax.yaxis.set_major_formatter(FuncFormatter(lambda v,_: f"{v:.1f}x"))
ax.legend(fontsize=9); ax.grid(True, axis="y", alpha=0.3)

for bar_grp in [b1, b2, b3]:
    for bar in bar_grp:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.02, f"{h:.2f}",
                ha="center", va="bottom", fontsize=7, color="#8B949E")

plt.tight_layout()
plt.savefig("fig_scenarios.png", dpi=150,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print("✅  Scenario chart saved → fig_scenarios.png")


## ⑬ AI regime detection overlay — illustrative path

In [ ]:
rp  = RESULTS[25]["ai_dynamic"]["regime_paths"][0]   # single path, shape (steps,)
fr_ai  = RESULTS[25]["ai_dynamic"]["funding_ratio"][0]
fr_cur = RESULTS[25]["current_75"]["funding_ratio"][0]

years = CFG["retire_age"] - 25
steps = int(years / CFG["dt"])
x_reg = np.linspace(0, years, steps)
x_fr  = np.linspace(0, years, steps+1)

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
fig.suptitle("AI Dynamic De-Risking — Illustrative Single Path (Entry Age 25)",
             fontsize=12, y=1.01)

# Panel 1: regime colour bands
for i in range(len(rp)-1):
    axes[0].axvspan(x_reg[i], x_reg[i+1],
                    alpha=0.35,
                    color=COL["cur"] if rp[i]==1 else COL["gs"],
                    lw=0)
axes[0].plot(x_reg, rp, color=COL["eq"], lw=1.5, alpha=0.9)
axes[0].set_yticks([0,1]); axes[0].set_yticklabels(["Bull/Normal","Bear/Stress"])
axes[0].set_ylabel("Regime")
axes[0].grid(False)
bull_p = mpatches.Patch(color=COL["gs"],  alpha=0.35, label="Normal / Bull regime")
bear_p = mpatches.Patch(color=COL["cur"], alpha=0.35, label="Stress / Bear regime")
axes[0].legend(handles=[bull_p, bear_p], fontsize=9, loc="upper left")
for sp in axes[0].spines.values(): sp.set_edgecolor(COL["ai"])

# Panel 2: funding ratio comparison
axes[1].plot(x_fr, fr_cur, color=COL["cur"],  lw=1.5, alpha=0.8, label="Current NPS (75% E)")
axes[1].plot(x_fr, fr_ai,  color=COL["ai"],   lw=2.2, alpha=0.95, label="AI Dynamic Glide Path")
axes[1].axhline(1.0, color="white", ls=":", lw=1.0, alpha=0.6, label="Target 1.0x")
axes[1].set_xlabel("Years from entry")
axes[1].set_ylabel("Funding Ratio")
axes[1].yaxis.set_major_formatter(FuncFormatter(lambda v,_: f"{v:.1f}x"))
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)
for sp in axes[1].spines.values(): sp.set_edgecolor(COL["ai"])

plt.tight_layout()
plt.savefig("fig_ai_overlay.png", dpi=150,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print("✅  AI overlay chart saved → fig_ai_overlay.png")


## ⑭ Interactive explorer (ipywidgets)
Use the dropdowns to explore any cohort × strategy combination instantly.

In [ ]:
def explore(entry_age, strategy):
    r   = RESULTS[entry_age][strategy]
    rr  = r["replacement_rate"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f"Age {entry_age} | {STRAT_LABELS[strategy]}", fontsize=11)

    # Left: histogram
    col = {"current_75":COL["cur"],"proposed":COL["prop"],"ai_dynamic":COL["ai"]}[strategy]
    axes[0].hist(np.clip(rr, 0, 7), bins=80, color=col, alpha=0.75, edgecolor="none")
    axes[0].axvline(np.median(rr), color="white", lw=2.0, ls="-",
                    label=f"Median {np.median(rr):.2f}x")
    axes[0].axvline(np.percentile(rr, 5), color="#E74C3C", lw=1.5, ls="--",
                    label=f"P5 {np.percentile(rr,5):.2f}x")
    axes[0].axvline(1.0, color="#F39C12", lw=1.5, ls=":",
                    label="Target 1.0x")
    axes[0].set_xlabel("Replacement Rate"); axes[0].set_ylabel("Frequency")
    axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

    # Right: funding ratio fan
    fr   = r["funding_ratio"]
    yrs  = CFG["retire_age"] - entry_age
    x    = np.linspace(0, yrs, fr.shape[1])
    for pct, alp in [(5,0.10),(25,0.22)]:
        axes[1].fill_between(x, np.percentile(fr,pct,axis=0),
                             np.percentile(fr,100-pct,axis=0),
                             alpha=alp, color=col)
    axes[1].plot(x, np.median(fr,axis=0), color=col, lw=2.0)
    axes[1].axhline(1.0, color="white", ls=":", lw=1.0, alpha=0.7)
    axes[1].set_xlabel("Years"); axes[1].set_ylabel("Funding Ratio")
    axes[1].yaxis.set_major_formatter(FuncFormatter(lambda v,_: f"{v:.1f}x"))
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()

    print(f"  Median RR : {r['median_rr']:.3f}x  |  "
          f"P5 : {r['p5_rr']:.3f}x  |  "
          f"P95 : {r['p95_rr']:.3f}x  |  "
          f"CVaR-95 : {r['cvar_95']:.3f}x  |  "
          f"Max DD : {r['drawdown_max']*100:.1f}%")

interact(explore,
         entry_age=widgets.Dropdown(options=ENTRY_AGES,   description="Entry Age:"),
         strategy =widgets.Dropdown(options=list(STRAT_LABELS.keys()),
                                    description="Strategy:",
                                    style={"description_width":"80px"}))


## ⑮ Export — standalone HTML report

In [ ]:
def fig_to_b64(path):
    """Read a saved PNG and return base64 string for HTML embedding."""
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def df_to_html(df, fmt_dict):
    """Render a DataFrame as a styled HTML table string."""
    styled = (df.style
              .format(fmt_dict)
              .set_table_styles([
                  {"selector":"th","props":[("background","#1F3864"),
                                             ("color","#E8D5A0"),
                                             ("padding","7px 14px"),
                                             ("font-size","11px")]},
                  {"selector":"tr:nth-child(even)","props":[("background","#EBF3FB")]},
                  {"selector":"td","props":[("padding","6px 14px"),
                                             ("font-size","11px")]},
              ])
              .hide(axis="index"))
    return styled.to_html()

RR_FMT = {"Median RR":"{:.2f}x","P5 RR":"{:.2f}x","P95 RR":"{:.2f}x",
           "CVaR 95%":"{:.2f}x","Max DD %":"{:.1f}%"}
SC_FMT = {"Median RR":"{:.2f}x","P5 RR":"{:.2f}x","P95 RR":"{:.2f}x",
          "CVaR 95%":"{:.2f}x","Max DD %":"{:.1f}%"}

# KPI values
best_rr  = RESULTS[25]["ai_dynamic"]["median_rr"]
curr_rr  = RESULTS[25]["current_75"]["median_rr"]
impr     = (best_rr/curr_rr - 1)*100
dd_curr  = RESULTS[25]["current_75"]["drawdown_max"]*100
dd_ai    = RESULTS[25]["ai_dynamic"]["drawdown_max"]*100
dd_red   = dd_curr - dd_ai

HTML_OUT = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>NPS Simulator — JCP 2026</title>
<link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@300;400;600&family=Spectral:ital,wght@0,300;0,600;1,300&display=swap" rel="stylesheet">
<style>
:root{{--bg:#0D1117;--surface:#161B22;--border:#30363D;--text:#E6EDF3;
       --subtext:#8B949E;--gold:#E8A838;--blue:#4A90D9;
       --green:#2ECC71;--red:#E74C3C;}}
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:'IBM Plex Mono',monospace;background:var(--bg);color:var(--text);font-size:13px;line-height:1.7}}
nav{{position:sticky;top:0;z-index:100;background:rgba(13,17,23,0.94);
     backdrop-filter:blur(12px);border-bottom:1px solid var(--border);
     padding:0 40px;display:flex;align-items:center;gap:28px;height:52px}}
.brand{{color:var(--gold);font-weight:600;font-size:13px;letter-spacing:.05em}}
nav a{{color:var(--subtext);text-decoration:none;font-size:11px;
       letter-spacing:.08em;text-transform:uppercase;transition:color .2s}}
nav a:hover{{color:var(--gold)}}
.hero{{background:linear-gradient(135deg,#0D1117 0%,#0f1e35 50%,#0D1117 100%);
       border-bottom:1px solid var(--border);padding:56px 40px 40px;position:relative;overflow:hidden}}
.hero::before{{content:"";position:absolute;top:-80px;right:-80px;width:450px;height:450px;
               background:radial-gradient(circle,rgba(232,168,56,.07) 0%,transparent 70%);pointer-events:none}}
.tag{{display:inline-block;background:rgba(232,168,56,.12);border:1px solid rgba(232,168,56,.35);
      color:var(--gold);font-size:10px;letter-spacing:.15em;text-transform:uppercase;
      padding:4px 12px;border-radius:2px;margin-bottom:16px}}
h1{{font-family:'Spectral',serif;font-size:28px;font-weight:600;line-height:1.3;max-width:700px;margin-bottom:12px}}
.hero p{{color:var(--subtext);font-size:12px;max-width:620px;line-height:1.8}}
.meta{{margin-top:24px;display:flex;gap:28px;flex-wrap:wrap}}
.meta-item{{color:var(--subtext);font-size:11px}}.meta-item strong{{color:var(--gold)}}
.kpi-grid{{display:grid;grid-template-columns:repeat(auto-fit,minmax(190px,1fr));
           gap:1px;background:var(--border);border:1px solid var(--border);margin:28px 40px}}
.kpi{{background:var(--surface);padding:18px 22px;transition:background .2s}}
.kpi:hover{{background:#1C2128}}
.kpi-label{{color:var(--subtext);font-size:10px;letter-spacing:.12em;text-transform:uppercase;margin-bottom:6px}}
.kpi-value{{font-size:26px;font-weight:600;letter-spacing:-.02em}}
.kpi-sub{{color:var(--subtext);font-size:10px;margin-top:3px}}
.section{{padding:36px 40px;border-bottom:1px solid var(--border)}}
.sec-title{{font-family:'Spectral',serif;font-size:19px;font-weight:600;
            margin-bottom:6px;display:flex;align-items:center;gap:10px}}
.sec-title::before{{content:"";display:block;width:3px;height:20px;
                    background:var(--gold);border-radius:2px}}
.sec-sub{{color:var(--subtext);font-size:11px;margin-bottom:22px;max-width:660px;line-height:1.8}}
.chart-box{{background:var(--surface);border:1px solid var(--border);
            border-radius:4px;padding:4px;margin-bottom:18px}}
.chart-box img{{width:100%;display:block;border-radius:2px}}
.cap{{color:var(--subtext);font-size:10px;text-align:center;padding:7px 10px 10px}}
.tbl-wrap{{overflow-x:auto;border:1px solid var(--border);border-radius:4px;margin-bottom:18px}}
footer{{padding:28px 40px;color:var(--subtext);font-size:10px;border-top:1px solid var(--border);
        display:flex;justify-content:space-between;flex-wrap:wrap;gap:14px}}
::-webkit-scrollbar{{width:5px;height:5px}}
::-webkit-scrollbar-track{{background:var(--bg)}}
::-webkit-scrollbar-thumb{{background:var(--border);border-radius:3px}}
</style></head><body>
<nav>
  <span class="brand">NPS // SIMULATOR</span>
  <a href="#mc">Monte Carlo</a><a href="#dist">Distributions</a>
  <a href="#sc">Scenarios</a><a href="#ai">AI Overlay</a>
</nav>
<div class="hero">
  <div class="tag">JCP 2026 · Track 2 · PFRDA / IAI</div>
  <h1>Optimal asset mix for NPS under long secular horizons</h1>
  <p>Monte Carlo simulation · Regime-switching GBM · Vasicek G-Sec model ·
     IAI LIC 2006-08 mortality · AI-driven dynamic de-risking</p>
  <div class="meta">
    <div class="meta-item"><strong>Generated</strong> · {datetime.datetime.now().strftime("%d %b %Y %H:%M")}</div>
    <div class="meta-item"><strong>Paths</strong> · {CFG['n_paths']:,}</div>
    <div class="meta-item"><strong>Cohorts</strong> · Age 25 / 35 / 45</div>
  </div>
</div>
<div class="kpi-grid">
  <div class="kpi"><div class="kpi-label">Median RR — Age 25 AI</div>
    <div class="kpi-value" style="color:var(--gold)">{best_rr:.2f}x</div>
    <div class="kpi-sub">AI Dynamic strategy</div></div>
  <div class="kpi"><div class="kpi-label">RR Improvement</div>
    <div class="kpi-value" style="color:var(--green)">+{impr:.1f}%</div>
    <div class="kpi-sub">vs current NPS 75% cap</div></div>
  <div class="kpi"><div class="kpi-label">Drawdown Reduction</div>
    <div class="kpi-value" style="color:var(--blue)">−{dd_red:.1f}pp</div>
    <div class="kpi-sub">AI vs current NPS</div></div>
  <div class="kpi"><div class="kpi-label">CVaR 95% — AI Age 25</div>
    <div class="kpi-value" style="color:#9B59B6">{RESULTS[25]["ai_dynamic"]["cvar_95"]:.2f}x</div>
    <div class="kpi-sub">Tail-risk metric</div></div>
  <div class="kpi"><div class="kpi-label">Median RR — Age 35 AI</div>
    <div class="kpi-value" style="color:var(--gold)">{RESULTS[35]["ai_dynamic"]["median_rr"]:.2f}x</div>
    <div class="kpi-sub">25-year horizon</div></div>
  <div class="kpi"><div class="kpi-label">Median RR — Age 45 AI</div>
    <div class="kpi-value" style="color:var(--blue)">{RESULTS[45]["ai_dynamic"]["median_rr"]:.2f}x</div>
    <div class="kpi-sub">15-year horizon</div></div>
</div>
<div id="mc" class="section">
  <div class="sec-title">Monte Carlo Funding Ratio Fan Charts</div>
  <div class="sec-sub">5–95th percentile corridor · 25–75th IQR band · Median path</div>
  <div class="chart-box"><img src="data:image/png;base64,{fig_to_b64('fig_fan_age25.png')}">
    <div class="cap">Fig 1a — Entry Age 25 (35-year horizon)</div></div>
  <div class="chart-box"><img src="data:image/png;base64,{fig_to_b64('fig_fan_age35.png')}">
    <div class="cap">Fig 1b — Entry Age 35 (25-year horizon)</div></div>
  <div class="chart-box"><img src="data:image/png;base64,{fig_to_b64('fig_fan_age45.png')}">
    <div class="cap">Fig 1c — Entry Age 45 (15-year horizon)</div></div>
</div>
<div id="dist" class="section">
  <div class="sec-title">Replacement Rate Distributions</div>
  <div class="sec-sub">Violin plots of terminal replacement rate — wider = more uncertainty</div>
  <div class="chart-box"><img src="data:image/png;base64,{fig_to_b64('fig_rr_distributions.png')}">
    <div class="cap">Fig 2 — RR violin plots: Current vs Proposed vs AI Dynamic</div></div>
  <div class="tbl-wrap">{df_to_html(df_cohort, RR_FMT)}</div>
</div>
<div class="section">
  <div class="sec-title">Glide Path Comparison</div>
  <div class="chart-box"><img src="data:image/png;base64,{fig_to_b64('fig_glide_paths.png')}">
    <div class="cap">Fig 3 — Current NPS (LC75) vs Proposed revised glide path</div></div>
</div>
<div id="sc" class="section">
  <div class="sec-title">Scenario Analysis</div>
  <div class="sec-sub">Six macroeconomic regimes · Age 25 · Proposed strategy</div>
  <div class="chart-box"><img src="data:image/png;base64,{fig_to_b64('fig_scenarios.png')}">
    <div class="cap">Fig 4 — Scenario analysis: Median RR / P5 RR / CVaR-95</div></div>
  <div class="tbl-wrap">{df_to_html(df_scenarios, SC_FMT)}</div>
</div>
<div id="ai" class="section">
  <div class="sec-title">AI Dynamic De-Risking Overlay</div>
  <div class="sec-sub">Regime classification + funding ratio: single illustrative path</div>
  <div class="chart-box"><img src="data:image/png;base64,{fig_to_b64('fig_ai_overlay.png')}">
    <div class="cap">Fig 5 — Bear/bull regime + AI vs Current NPS funding ratio trajectory</div></div>
</div>
<footer>
  <div style="max-width:580px;line-height:1.7">
    <strong style="color:var(--text)">Disclaimer:</strong> For academic research purposes —
    JCP 2026 paper submission to PFRDA &amp; IAI. Not investment advice.
    Paper: <em>Optimal asset mix for NPS under long secular horizons</em>.
  </div>
  <div>NPS Simulator v1.0 · Python · NumPy · scikit-learn · Matplotlib<br>
  Generated: {datetime.datetime.now().strftime("%d %b %Y %H:%M IST")}</div>
</footer>
</body></html>"""

with open("nps_report.html","w",encoding="utf-8") as f:
    f.write(HTML_OUT)

print("✅  HTML report saved → nps_report.html")
print()
print("Open in browser with:")
print('  import webbrowser; webbrowser.open("nps_report.html")')
print()
print("Or display inline:")
print('  IFrame("nps_report.html", width=1200, height=700)')


## ⑯ View report inline in Jupyter

In [ ]:
# Inline preview
display(IFrame("nps_report.html", width="100%", height=800))


## ⑰ Export all data tables to Excel

In [ ]:
with pd.ExcelWriter("NPS_DataTables.xlsx", engine="openpyxl") as writer:
    # T1: Capital market assumptions
    pd.DataFrame([
        ["Equity (E)",          0.135, 0.180, -0.080, 0.380],
        ["Corporate Bonds (C)", 0.080, 0.045,  None,  None ],
        ["Govt Securities (G)", 0.072, 0.035,  None,  None ],
        ["Alt / InvIT",         0.105, 0.120,  None,  None ],
        ["Inflation (CPI)",     0.055, 0.015,  None,  None ],
        ["Real Salary Growth",  0.020, None,   None,  None ],
    ], columns=["Asset Class","Bull Return","Bull Vol","Bear Return","Bear Vol"]
    ).to_excel(writer, sheet_name="T1_CapMktAssumptions", index=False)

    # T2: Simulation results
    df_cohort.to_excel(writer, sheet_name="T2_SimResults", index=False)

    # T3: Scenario results
    df_scenarios.to_excel(writer, sheet_name="T3_ScenarioResults", index=False)

    # T4: Glide paths
    gp_rows = []
    for a in range(25, 61):
        wc = glide_weights("current_75", a)
        wp = glide_weights("proposed",   a)
        gp_rows.append([a, wc["E"],wc["C"],wc["G"],wc["Alt"],
                           wp["E"],wp["C"],wp["G"],wp["Alt"]])
    pd.DataFrame(gp_rows, columns=[
        "Age","Curr_E","Curr_C","Curr_G","Curr_Alt",
        "Prop_E","Prop_C","Prop_G","Prop_Alt"]
    ).to_excel(writer, sheet_name="T4_GlidePaths", index=False)

    # T5: Mortality (selected ages)
    sel = [25,30,35,40,45,50,55,60,65,70,75,80]
    pd.DataFrame({
        "Age x" : sel,
        "qx"    : [round(QX[a],6) for a in sel],
        "lx"    : [round(LX[a],1) for a in sel],
        "10px"  : [round(tpx(a,10),4) for a in sel],
        "ann_7.2%": [round(annuity_pv(a,0.072),3) for a in sel],
    }).to_excel(writer, sheet_name="T5_Mortality", index=False)

print("✅  Excel file saved → NPS_DataTables.xlsx")
